In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import AGEMTrainer, IntervalAGEMTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils.general import InContextHead
from src import models
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

from configs import MNIST_AGEM_CONFIG as CONFIG

### Task Incremental Learning

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda", seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)


In [ ]:
agem_trainer = AGEMTrainer(
    model,
    memory_samples=100,
    paradigm="TIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    agem_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        context_id=i,
    )
    agem_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))

### Domain Incremental Learning

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=2, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [ ]:
agem_trainer = AGEMTrainer(
    model,
    memory_samples=100,
    paradigm="DIL",
    domain_map_fn=domain_map_fn,
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    agem_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    agem_trainer.test(test_tasks)

### Class Incremental Training

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
agem_trainer = AGEMTrainer(
    model,
    memory_samples=200,
    paradigm="CIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    agem_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    agem_trainer.test(test_tasks)

## IntervalAGEMTrainer

### Class Incremental Learning

In [ ]:
from configs import MNIST_IT_CONFIG as INTERVAL_CONFIG

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
agem_trainer = IntervalAGEMTrainer(
    model,
    memory_samples=200,
    checkpoint=INTERVAL_CONFIG["checkpoint"],
    n_iters=INTERVAL_CONFIG["n_iters"],
    min_acc_limit=INTERVAL_CONFIG["min_acc_limit"],
    min_acc_increment=INTERVAL_CONFIG["min_acc_increment"],
    primal_learning_rate=INTERVAL_CONFIG["primal_learning_rate"],
    dual_learning_rate=INTERVAL_CONFIG["dual_learning_rate"],
    projection_strategy=INTERVAL_CONFIG["projection_strategy"],
    n_certificate_samples=INTERVAL_CONFIG["n_certificate_samples"],
    penalty_coefficient=INTERVAL_CONFIG["penalty_coefficient"],
    paradigm="CIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    agem_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    agem_trainer.test(test_tasks)
    if i < len(test_tasks) - 1:
        agem_trainer.compute_rashomon_set(test)

# Ablation Study

In [4]:
import wandb

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x70f8f2cf5e40>>
Traceback (most recent call last):
  File "/vol/bitbucket/le24/.venv/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [ ]:
# HYPERPARAMS
""" TIL
MNIST = {
    "batch_size": 256,
    "epochs": 3,
    "lr": 0.001,
    "weight_decay": 0,
    "unbias_lambda": 0.01,
    "l2_lambda": 0.01
}
"""

for paradigm in ["TIL", "DIL", "CIL"]:
    for i in range(5):
        failed = False
        with wandb.init(
            project="certified-continual-learning",
            reinit=True,
            tags=["final_mnist_buffer", "buffer_agem", f"buffer_{paradigm.lower()}"],
        ):
            def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
                """Map the global label to the in context label."""
                return labels % 2
            wandb.log({"seed": i})
            SEED = i
            train_tasks, _, test_tasks = get_mnist_tasks(seed=SEED, train_val_split_ratio=0.3, emnist=True)
            context_sets = get_context_sets(test_tasks)
            head = InContextHead(context_sets, 10, device="cuda")
            head.set_context(0)
            model = models.get_mnist_model(device="cuda", output_dim=2 if paradigm == "DIL" else 10, seed=SEED, head = head if paradigm=="TIL" else None)
            print(
                f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
            )

            agem_trainer = AGEMTrainer(
                model,
                memory_samples=3750,
                paradigm=paradigm,
                seed=SEED,
                domain_map_fn=domain_map_fn if paradigm == "DIL" else None
            )

            acc_matrix = []
            for i, (train, test) in enumerate(
                zip(train_tasks, test_tasks)
            ):
                agem_trainer.train(
                    train,
                    test,
                    batch_size=CONFIG["batch_size"],
                    epochs=CONFIG["epochs"],
                    lr=CONFIG["lr"],
                    weight_decay=CONFIG["weight_decay"],
                    context_id=i if paradigm == "TIL" else None,
                    val_freq=int(len(train) / CONFIG["batch_size"]) - 1
                )
                results = agem_trainer.test(
                    test_tasks,
                    context_list=list(range(len(test_tasks)))
                    if paradigm == "TIL"
                    else [None] * len(test_tasks),
                )
                accs = [res[1] for res in results]
                if not i and accs[0] < 0.7:
                    wandb.finish(1)
                    failed = True
                    break
                acc_matrix.append(accs)

            if not failed:
                columns = [f"Test Task {i}" for i in range(len(test_tasks))]
                rows = [f"Task {i}" for i in range(len(test_tasks))]
                wandb.log(
                    {
                        "accuracy_matrix": wandb.Table(
                            data=acc_matrix, columns=columns, rows=rows
                        )
                    }
                )
                wandb.finish()

### CIFAR

In [11]:
from collections import defaultdict
from src.helpers.WandbWrapper import WandbTrainerWrapper
from src.models import get_mnist_model
from src.data_utils import get_mnist_tasks
from src.utils.general import set_seed
import torchvision
import wandb

results = {
    "class": defaultdict(list),
    "task": defaultdict(list),
    "domain": defaultdict(list),
}

import copy
import sys
import random
import torchvision
import torchvision.transforms as transforms
import torch
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)
from src.trainer import IntervalTrainer, InterContiNetTrainer

# Define transform (e.g., convert to tensor)
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3),
    ]
)

from src.data_utils import split_mnist_by_labels, get_context_sets
from src.utils import general as utils
import src.models as models

import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json

from src.utils.plotting_style import set_figure_size
from src.regulariser import L2Regulariser, UnbiasRegulariser, MultiRegulariser

In [12]:
device = "cuda"
classes = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

task_pairs = [
    ("cat", "truck"),
    ("frog", "ship"),
    ("horse", "automobile"),
    ("dog", "airplane"),
    ("bird", "deer"),
]
task_pairs_ids = [(classes.index(i), classes.index(j)) for i, j in task_pairs]
animals_mask = torch.tensor([0, 0, 1, 1, 1, 1, 1, 1, 0, 0]).to(device)


def domain_map_fn(y):
    return animals_mask[y]


@torch.no_grad()
def encode(x, model, device="cuda"):
    # Handle batching to avoid out-of-memory issues
    batch_size = 2048
    features = []
    for i in range(0, len(x), batch_size):
        batch = x[i : i + batch_size].to(device)
        batch_features = model(batch).flatten(start_dim=1).cpu()
        features.append(batch_features)
    return torch.cat(features, dim=0)


def encode_dataset(train_dataset, test_dataset, encoder, device="cuda"):
    train_imgs, train_labels = train_dataset.data, train_dataset.targets
    test_imgs, test_labels = test_dataset.data, test_dataset.targets
    # apply the appropriate scaling and transposition
    train_imgs = (
        torch.tensor(train_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    )
    test_imgs = torch.tensor(test_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    train_imgs = (train_imgs - 0.5) / 0.5
    test_imgs = (test_imgs - 0.5) / 0.5
    train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
    test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()

    if encoder is not None:
        train_imgs = encode(train_imgs, encoder, device)
        test_imgs = encode(test_imgs, encoder, device)

    train_dataset = torch.utils.data.TensorDataset(train_imgs, train_labels)
    test_dataset = torch.utils.data.TensorDataset(test_imgs, test_labels)
    return train_dataset, test_dataset


def get_tasks(encoder, seed = 42):
    set_seed(seed)
    non_animals = [0, 1, 8, 9]
    animals = [2, 3, 4, 5, 6, 7]

    non_animal_indices = torch.tensor(non_animals)[torch.randperm(4)].tensor_split(5)
    animal_indices = list(torch.tensor(animals)[torch.randperm(6)].tensor_split(5))
    animal_indices.reverse()

    task_pairs_ids = [
        t.tolist() + n.tolist() for t, n in zip(animal_indices, non_animal_indices)
    ]
    print("Contexts:", task_pairs_ids)
    trainset = torchvision.datasets.CIFAR10(
        root="./data", train=True, download=True, transform=transform
    )
    testset = torchvision.datasets.CIFAR10(
        root="./data", train=False, download=True, transform=transform
    )
    trainset.targets = torch.tensor(trainset.targets)
    testset.targets = torch.tensor(testset.targets)

    if encoder is not None:
        trainset, testset = encode_dataset(trainset, testset, encoder)
    test_tasks = [
        split_mnist_by_labels(testset, task_pair_id) for task_pair_id in task_pairs_ids
    ]
    train_tasks = [
        split_mnist_by_labels(trainset, task_pair_id) for task_pair_id in task_pairs_ids
    ]

    return train_tasks, test_tasks

In [13]:
# Get the CIFAR 100 dataset
cifar100_trainset = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=True, transform=transform
)
cifar100_testset = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=transform
)

# Convert targets to tensor for consistency
cifar100_trainset.targets = torch.tensor(cifar100_trainset.targets)
cifar100_testset.targets = torch.tensor(cifar100_testset.targets)

# Print dataset info
print(f"CIFAR-100 training set: {len(cifar100_trainset)} images")
print(f"CIFAR-100 test set: {len(cifar100_testset)} images")
print(f"Number of classes: {len(set(cifar100_trainset.targets.tolist()))}")

# Sample a few class names
print(f"Sample classes: {cifar100_trainset.classes[:10]}")

CIFAR-100 training set: 50000 images
CIFAR-100 test set: 10000 images
Number of classes: 100
Sample classes: ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle']


In [14]:
# Create a simple function to load the model
save_dir = os.path.join(project_root, "notebooks/saved_models")
model_path = os.path.join(save_dir, "resnet18_cifar100_pretrained.pth")


def load_pretrained_model(model_path, num_classes=100):
    model = torchvision.models.resnet18(weights=None)
    model.fc = torch.nn.Linear(512, num_classes)
    model.load_state_dict(torch.load(model_path))
    return model


model = load_pretrained_model(model_path)

In [15]:
encoder = copy.deepcopy(model).cuda()
encoder.fc = torch.nn.Identity()

In [16]:
X_min, X_max = None, None
for task in get_tasks(encoder):
    X, _ = next(
        iter(torch.utils.data.DataLoader(task[0], batch_size=10000, shuffle=False))
    )
    if X_min is None:
        X_min, X_max = X.min(dim=0).values, X.max(dim=0).values
    else:
        X_min = torch.min(X_min, X.min(dim=0).values)
        X_max = torch.max(X_max, X.max(dim=0).values)
X_min, X_max = X_min.to(device), X_max.to(device)

Contexts: [[3, 8], [5, 9], [2, 0], [7, 1], [6, 4]]


/tmp/ipykernel_1424862/3250057340.py:52: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_1424862/3250057340.py:53: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()


In [ ]:
# HYPERPARAMS
""" TIL
MNIST = {
    "batch_size": 256,
    "epochs": 3,
    "lr": 0.001,
    "weight_decay": 0,
    "unbias_lambda": 0.01,
    "l2_lambda": 0.01
}
"""

for paradigm in ["DIL"]:
    for i in range(5, 15):
        failed = False
        with wandb.init(
            project="certified-continual-learning",
            reinit=True,
            tags=["final_cifar_buffer", "buffer_agem", f"buffer_{paradigm.lower()}"],
        ):
            wandb.log({"seed": i})
            SEED = i
            set_seed(i)
            train_tasks, test_tasks = get_tasks(encoder, seed=SEED)
            context_sets = get_context_sets(test_tasks)
            head = InContextHead(context_sets, 10, device="cuda")
            head.set_context(0)
            model = models.get_fully_connected_model(
                input_dim=512, seed=SEED, device="cuda", output_dim=10 if paradigm != "DIL" else 2, head=head if paradigm=="TIL" else None, 
            )
            print(
                f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
            )

            agem_trainer = AGEMTrainer(
                model,
                memory_samples=3750,
                paradigm=paradigm,
                seed=SEED,
                domain_map_fn=domain_map_fn if paradigm == "DIL" else None
            )

            acc_matrix = []
            for i, (train, test) in enumerate(
                zip(train_tasks, test_tasks)
            ):
                agem_trainer.train(
                    train,
                    test,
                    batch_size=CONFIG["batch_size"],
                    epochs=CONFIG["epochs"],
                    lr=CONFIG["lr"],
                    weight_decay=CONFIG["weight_decay"],
                    context_id=i if paradigm == "TIL" else None,
                    val_freq=int(len(train) / CONFIG["batch_size"]) - 1
                )
                results = agem_trainer.test(
                    test_tasks,
                    context_list=list(range(len(test_tasks)))
                    if paradigm == "TIL"
                    else [None] * len(test_tasks),
                )
                accs = [res[1] for res in results]
                if not i and accs[0] < 0.65:
                    wandb.finish(1)
                    failed = True
                    break
                acc_matrix.append(accs)

            if not failed:
                columns = [f"Test Task {i}" for i in range(len(test_tasks))]
                rows = [f"Task {i}" for i in range(len(test_tasks))]
                wandb.log(
                    {
                        "accuracy_matrix": wandb.Table(
                            data=acc_matrix, columns=columns, rows=rows
                        )
                    }
                )
                wandb.finish()

wandb: Currently logged in as: le24 (agt-team) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Contexts: [[2, 9], [7, 1], [3, 0], [6, 8], [5, 4]]


/tmp/ipykernel_1424862/3250057340.py:52: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_1424862/3250057340.py:53: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()


Tasks: [[2, 9], [1, 7], [0, 3], [6, 8], [4, 5]]


Training Epochs: 100%|███████████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  2.67it/s, loss=0.4489, acc=0.7500]


...

Test Results: [(0.5284, 0.759), (0.3297, 0.859), (0.9684, 0.5685), (0.6418, 0.68), (0.5546, 0.78)] (Avg: (0.6046, 0.7293))


Training Epochs:   0%|                                                                                       | 0/3 [00:03<?, ?it/s, loss=0.4485, acc=0.8008]Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x70f8f2cf5e40>>
Traceback (most recent call last):
  File "/vol/bitbucket/le24/.venv/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
Training Epochs: 100%|███████████████████████████████████████████████████████████████████████████████| 3/3 [00:12<00:00,  4.13s/it, loss=0.2846, acc=0.8125]


Test Results: [(0.7433, 0.6565), (0.5928, 0.703), (0.3226, 0.863), (0.3766, 0.846), (0.3929, 0.8445)] (Avg: (0.4856, 0.7826))


Training Epochs:   0%|                                                                                         | 0/3 [00:03<?, ?it/s, loss=0.2236, acc=None]

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x70f8f2cf5e40>>
Traceback (most recent call last):
  File "/vol/bitbucket/le24/.venv/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
Training Epochs:  33%|██████████████████████████▎                                                    | 1/3 [00:06<00:12,  6.15s/it, loss=0.2013, acc=0.9219]